# 03. Paneles y merge

Arma los paneles de trabajo a partir de las bases de 01 y 02.

| Panel | Unidad | Archivo |
|---|---|---|
| Clase × día | clase de fondo MM en pesos, día | `panel_clase_dia.parquet` |
| Fondo × día | fondo (código CNV), día | `panel_fondo_dia.parquet` |
| Fondo × semana, planilla + cartera | fondo, semana (cierre viernes) | `panel_fondo_semana.parquet` / `.dta` |

**Variables derivadas** (definiciones completas en `claude/diccionario_fci.md`):
- `r_neto` = VCP / VCP del día anterior − 1; `tna_neta` = r_neto / días × 365, en %;
- `fee` = honorarios SG + SD + gastos, en % anual; `tna_bruta` = tna_neta + fee. Es lo que rinde la cartera y es igual entre clases del mismo fondo;
- `flujo` = (cuotapartes − cuotapartes del día anterior) × VCP / 1.000, en pesos. Mide suscripciones netas sin mezclar rendimiento.

El merge usa sólo las filas del crosswalk con `revisado = 1`.

In [ ]:
import sys, importlib
from pathlib import Path
import numpy as np
import pandas as pd

# el módulo vive en esta misma carpeta
sys.path.insert(0, str(Path.cwd()))
import fondoscnv as fc
importlib.reload(fc)

R = fc.rutas()          # raíz del repo y carpetas de datos
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
{k: str(v) for k, v in R.items()}

## 1. Clase × día y fondo × día

In [ ]:
mm = pd.read_parquet(R["proc"] / "planillas_mm.parquet")
cd = fc.derivar_clase_dia(mm)
cd.to_parquet(R["proc"] / "panel_clase_dia.parquet", index=False)

fd = fc.fondo_dia(cd)
fd.to_parquet(R["proc"] / "panel_fondo_dia.parquet", index=False)
print(f"clase x día: {len(cd):,} | fondo x día: {len(fd):,}")
cd[["tna_neta", "tna_bruta", "fee", "dias"]].describe().T

In [ ]:
# chequeo central: dentro de un fondo y un día, la TNA bruta debería ser igual entre clases
disp = (cd.dropna(subset=["tna_bruta"])
          .groupby(["cod_cnv", "fecha"])["tna_bruta"].agg(["std", "count"]))
disp = disp[disp["count"] > 1]
print("desvío intra-fondo de la TNA bruta (pp):")
disp["std"].describe(percentiles=[.5, .9, .99])

## 2. Fondo × semana (planilla)

In [ ]:
fs = fc.fondo_semana(fd)
fs.to_parquet(R["proc"] / "panel_fondo_semana_planilla.parquet", index=False)
fs.tail()

## 3. Cartera por fondo y fecha, en formato ancho

In [ ]:
E = pd.read_parquet(R["proc"] / "carteras_encabezado.parquet")
L = pd.read_parquet(R["proc"] / "carteras_lineas.parquet")
ca = fc.cartera_ancha(E, L)
ca.to_parquet(R["proc"] / "carteras_ancha.parquet", index=False)
ca[["lfc", "fondo_cnv", "cartera_al", "pct_disponibilidades", "pct_plazo_fijo",
    "pct_caucion", "pct_titulos", "pct_pasivos", "n_bancos_pf", "pn_cartera"]].tail()

## 4. Merge planilla + cartera

La llave es (código CNV ↔ id de ficha, semana). Si un fondo tiene más de una cartera en la misma semana, queda la última. La cartera se toma de la semana en que está fechada ("Cartera al"), no de la fecha pedida.

In [ ]:
xw = pd.read_csv(R["xwalk"] / "fondos_cnv_lfc.csv")
xw = xw[xw["revisado"] == 1][["cod_cnv", "lfc"]].astype({"lfc": str})
print("fondos con crosswalk revisado:", len(xw))

ca1 = (ca.sort_values("cartera_al").drop_duplicates(["lfc", "semana"], keep="last")
         .astype({"lfc": str}).merge(xw, on="lfc", how="inner"))
panel = fs.merge(ca1, on=["cod_cnv", "semana"], how="left", suffixes=("", "_cartera"))
panel["tiene_cartera"] = panel["cartera_al"].notna().astype(int)
panel["pn_ratio"] = panel["pn"] / panel["pn_cartera"]
print(f"fondo x semana: {len(panel):,} | con cartera: {panel['tiene_cartera'].mean():.1%}")
panel.groupby(panel["semana"].dt.year)["tiene_cartera"].mean().rename("share con cartera").to_frame().T

In [ ]:
# el patrimonio de la planilla y el de la cartera deberían estar cerca
panel.loc[panel["tiene_cartera"] == 1, "pn_ratio"].describe(percentiles=[.05, .5, .95])

In [ ]:
panel.to_parquet(R["proc"] / "panel_fondo_semana.parquet", index=False)

# versión Stata: strings como object y nombres de hasta 32 caracteres
st = panel.copy()
for c in st.columns:
    if str(st[c].dtype) in ("string", "object"):
        st[c] = st[c].astype(object).where(st[c].notna(), "").astype(str)
st.columns = [c[:32] for c in st.columns]
st.to_stata(R["proc"] / "panel_fondo_semana.dta", write_index=False, version=118)
print("guardado:", R["proc"] / "panel_fondo_semana.dta")

## 5. Primera mirada: share en disponibilidades alrededor de los encajes

Promedio ponderado por patrimonio, fondos con cartera. Las líneas verticales marcan las fechas de vigencia de los cambios de encaje sobre la vista de los FCI money market.

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({"font.family": "serif", "font.size": 10,
                     "axes.spines.top": False, "axes.spines.right": False})

ENCAJES = {"A 7988": "2024-04-15", "A 8000": "2024-05-15", "A 8119": "2024-11-01",
           "A 8281/8289": "2025-08-01", "A 8355": "2025-12-01"}

x = panel[panel["tiene_cartera"] == 1].copy()
x["w"] = x["pn"]
serie = (x.groupby("semana")
           .apply(lambda g: pd.Series({
               "disp": np.average(g["pct_disponibilidades"].fillna(0), weights=g["w"]),
               "pf": np.average(g["pct_plazo_fijo"].fillna(0), weights=g["w"]),
               "cau": np.average(g["pct_caucion"].fillna(0), weights=g["w"])})))

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.plot(serie.index, serie["disp"], color="black", lw=1.2, label="Disponibilidades (vista)")
ax.plot(serie.index, serie["pf"], color="0.45", lw=1.0, ls="--", label="Plazo fijo")
ax.plot(serie.index, serie["cau"], color="0.65", lw=1.0, ls=":", label="Caución")
for k, v in ENCAJES.items():
    ax.axvline(pd.Timestamp(v), color="0.6", lw=0.6)
    ax.text(pd.Timestamp(v), ax.get_ylim()[1], k, rotation=90, va="top", ha="right", fontsize=7, color="0.4")
ax.set_ylabel("% del patrimonio")
ax.legend(frameon=False, fontsize=8, loc="upper left")
fig.tight_layout()
out = R["root"] / "Code" / "output" / "figuras"
out.mkdir(parents=True, exist_ok=True)
fig.savefig(out / "fci_mm_composicion_semanal.pdf")
plt.show()